In [38]:
import torch
import torch.nn as nn
import torch.optim as optim
from rich.progress import track
import numpy as np
import torch.nn.functional as F

In [5]:
sentences = [
    # --- Simple "I love/like" Declarations ---
    "i love code",
    "i like math",
    "i love data",
    "i like stats",

    # --- Technology & Concept Statements ---
    "data is clean",
    "ai is fast",
    "ml is hard",
    "code is power",

    # --- Direct Concept Questions ---
    "what is chat",
    "what is deep",
    "where is tech",
    "how is code",

    # --- Conversational AI Questions ---
    "who am i",
    "what are you",
    "can you code"
]

In [9]:
from collections import Counter

words = " ".join(sentences).split()
vocab = {word:i for i, word in enumerate(set(words))}

In [11]:
def encode(sentence):
    return [vocab[word] for word in sentence.split()]

encoded = [encode(sentence) for sentence in sentences]

In [31]:
X = []
y = []

for chunk in encoded:
  X.append(chunk[:-1])
  y.append(chunk[1:])

X = torch.tensor(X)
y = torch.tensor(y)

print(X)
# print(y)

tensor([[17, 15],
        [17,  9],
        [17, 15],
        [17,  9],
        [ 8, 22],
        [23, 22],
        [ 0, 22],
        [19, 22],
        [20, 22],
        [20, 22],
        [ 1, 22],
        [ 5, 22],
        [14,  7],
        [20, 13],
        [16,  3]])


In [18]:
class NextRNN(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, 10)
    self.rnn = nn.RNN(10, 10, batch_first=True)
    self.linear = nn.Linear(10, vocab_size)

  def forward(self,x):
    x = self.embedding(x)
    out, _ = self.rnn(x)
    out = out[:,-1,:]
    out = self.linear(out)

    return out

In [19]:
model = NextRNN(len(vocab))

loss_fn = nn.CrossEntropyLoss()
optim = optim.Adam(model.parameters(),lr=0.01)

In [27]:
for epoch in track(range(200),description="Training Data:"):
  output = model.forward(X)

  loss = loss_fn(output, y[:, -1])
  optim.zero_grad()

  loss.backward()
  optim.step()
  if not epoch%20:
    # print(loss.item())
    pass

Output()

In [60]:
test_X = input('enter text: ')

model.eval()

with torch.no_grad():
  data = [vocab[chunk] for chunk in test_X.split()]
  data = torch.tensor([data])

  output = model(data)
  word = F.softmax(output, dim=-1).argmax(dim=-1)
  word = list(vocab.keys())[list(vocab.values()).index(word.item())]

  print(word)

  print(model.embedding(data))

enter text: i
hard
tensor([[[-0.8460, -0.1506, -0.9318, -0.1369, -3.4398,  0.7910, -0.4422,
          -1.7084, -0.3603, -0.7430]]])
